In [1]:
import duckdb

# Connect to an in-memory DuckDB instance
con = duckdb.connect()



In [2]:
# Stream CSV directly into compressed Parquet files
files = ["mobile_app_usage_data_1", "mobile_app_usage_data_2"]

for f in files:
    raw_path = f"data/raw/{f}.csv"
    parquet_path = f"data/parquet/{f}.parquet"

    query = f"""
    COPY (SELECT * FROM read_csv_auto('{raw_path}'))
    TO '{parquet_path}' (FORMAT PARQUET, CODEC 'ZSTD');
    """
    con.execute(query)
    print(f"Converted: {raw_path} -> {parquet_path}")

Converted: data/raw/mobile_app_usage_data_1.csv -> data/parquet/mobile_app_usage_data_1.parquet
Converted: data/raw/mobile_app_usage_data_2.csv -> data/parquet/mobile_app_usage_data_2.parquet


In [3]:
import os

# Check compressed file sizes
for f in ["mobile_app_usage_data_1", "mobile_app_usage_data_2"]:
    raw_sz = os.path.getsize(f"data/raw/{f}.csv") / (1024**3)
    pq_sz = os.path.getsize(f"data/parquet/{f}.parquet") / (1024**3)
    print(f"{f}: {raw_sz:.2f} GB (CSV) -> {pq_sz:.2f} GB (Parquet)")

# Verify total combined row count
count = con.execute(
    "SELECT count(*) FROM 'data/parquet/*.parquet'"
).fetchone()[0]
print(f"\nTotal combined rows: {count:,}")

mobile_app_usage_data_1: 23.20 GB (CSV) -> 1.25 GB (Parquet)
mobile_app_usage_data_2: 23.29 GB (CSV) -> 1.25 GB (Parquet)

Total combined rows: 179,921,470


In [16]:
# Raw events view
con.execute(
    """
    CREATE OR REPLACE VIEW raw_usage AS 
    SELECT * FROM 'data/parquet/mobile_app_usage_data_*.parquet';
"""
)

# Aggregated metrics view
con.execute(
    """
    CREATE OR REPLACE VIEW daily_metrics AS 
    SELECT * FROM 'data/processed/daily_app_metrics.parquet';
"""
)

In [17]:
import os

processed_size = (
    os.path.getsize("data/processed/daily_app_metrics.parquet") / (1024**2)
)
print(f"Processed file size: {processed_size:.2f} MB")

# Preview top aggregated rows
con.execute("SELECT * FROM daily_metrics LIMIT 5;").df()

Processed file size: 65.87 MB


,event_date,app_name,country,access_platform,is_premium_flag,active_users,session_count,total_events,total_duration_min,avg_session_min,crashed_sessions
0,2028-03-03,VideoStreaming,India,Opera,Y,7,7,20,1294.0,64.70,4
1,2029-01-05,PlayStore,India,Safari,N,10,19,41,1591.0,38.80,12
2,2027-11-11,Health,UK,Edge,N,4,7,21,1142.0,54.38,3
3,2029-07-14,Adobe,Russia,Firefox,N,6,11,28,4889.0,174.61,1
4,2029-07-18,ShareChat,Singapore,Safari,Y,9,16,41,16164.0,394.24,9


In [5]:
con.execute("DESCRIBE raw_usage;").df()

,column_name,column_type,null,key,default,extra
0,user_id,BIGINT,YES,None,None,None
1,first_login_at,TIMESTAMP,YES,None,None,None
2,last_active_at,TIMESTAMP,YES,None,None,None
3,registration_source,VARCHAR,YES,None,None,None
4,country,VARCHAR,YES,None,None,None
5,is_premium_flag,VARCHAR,YES,None,None,None
6,device_id,BIGINT,YES,None,None,None
7,access_platform,VARCHAR,YES,None,None,None
8,device_model,VARCHAR,YES,None,None,None
9,first_seen_at,TIMESTAMP,YES,None,None,None


In [6]:
con.execute("SELECT * FROM raw_usage LIMIT 5;").df()

,user_id,first_login_at,last_active_at,registration_source,country,is_premium_flag,device_id,access_platform,device_model,first_seen_at,...,app_name,session_id,session_start_time,duration_min,session_end_time,has_crash_ended_flag,event_id,event_name,event_sequence_in_session,crash_time
0,1,2029-12-26 02:06:18.338426,2030-01-30 02:06:18.338426,yahoo_auth,Russia,N,1,Chrome,Lenovo,2030-01-26 18:24:18.338426,...,Snapchat,VCV-9676-HNNL-63142,2030-01-26 06:34:18.338426,465,2030-01-26 14:19:18.338426,N,1,screen_view,1,NaT
1,1,2029-12-26 02:06:18.338426,2030-01-30 02:06:18.338426,yahoo_auth,Russia,N,1,Chrome,Lenovo,2030-01-26 18:24:18.338426,...,Snapchat,VCV-9676-HNNL-63142,2030-01-26 06:34:18.338426,465,2030-01-26 14:19:18.338426,N,2,screen_view,2,NaT
2,1,2029-12-26 02:06:18.338426,2030-01-30 02:06:18.338426,yahoo_auth,Russia,N,1,Chrome,Lenovo,2030-01-26 18:24:18.338426,...,Snapchat,VCV-9676-HNNL-63142,2030-01-26 06:34:18.338426,465,2030-01-26 14:19:18.338426,N,3,screen_view,3,NaT
3,1,2029-12-26 02:06:18.338426,2030-01-30 02:06:18.338426,yahoo_auth,Russia,N,2,Chrome,HP,2030-01-22 18:59:18.338426,...,MSExcel,IYM-0029-ACSQ-29317,2030-01-30 01:41:18.338426,52,2030-01-30 02:33:18.338426,Y,1,start_button_click,1,2030-01-30 02:33:18.338426
4,1,2029-12-26 02:06:18.338426,2030-01-30 02:06:18.338426,yahoo_auth,Russia,N,2,Chrome,HP,2030-01-22 18:59:18.338426,...,MSExcel,IYM-0029-ACSQ-29317,2030-01-30 01:41:18.338426,52,2030-01-30 02:33:18.338426,Y,2,start_button_click,2,2030-01-30 02:33:18.338426


In [7]:
quality_query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT session_id) AS unique_sessions,
    COUNT(DISTINCT app_name) AS unique_apps,
    MIN(session_start_time) AS min_session_start,
    MAX(session_start_time) AS max_session_start,
    COUNT(*) FILTER (WHERE user_id IS NULL) AS null_user_ids,
    COUNT(*) FILTER (WHERE session_id IS NULL) AS null_session_ids,
    COUNT(*) FILTER (WHERE duration_min < 0) AS negative_durations
FROM raw_usage;
"""

con.execute(quality_query).df()

,total_records,unique_users,unique_sessions,unique_apps,min_session_start,max_session_start,null_user_ids,null_session_ids,negative_durations
0,179921470,1999998,71971418,29,2024-12-27 05:08:51.444254,2031-03-06 22:46:26.462038,0,0,0


In [8]:
# -- 1. App popularity & engagement metrics
popularity = """SELECT 
    app_name,
    COUNT(DISTINCT user_id) AS monthly_active_users,
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(*) AS total_events,
    ROUND(AVG(duration_min), 2) AS avg_session_duration_min
FROM raw_usage
GROUP BY app_name
ORDER BY monthly_active_users DESC;"""

con.execute(popularity).df()


,app_name,monthly_active_users,total_sessions,total_events,avg_session_duration_min
0,WhatsApp,259001,2498377,6244330,349.98
1,Weather,258453,2490976,6227643,47.50
2,MSPowerPoint,257999,2482587,6206481,179.97
3,Snapchat,257879,2493025,6234444,349.95
4,VideoStreaming,257765,2486323,6215489,47.50
5,Adobe,257746,2481857,6205250,179.97
6,Flipkart,257716,2484127,6214346,47.49
7,VoiceRecorder,257711,2483777,6210302,47.48
8,TikTok,257700,2481483,6203255,350.05
9,Walmart,257694,2485377,6212645,47.50


In [9]:

# -- 2. Platform breakdown (iOS vs Android vs Web)
platform = """
SELECT 
    access_platform,
    COUNT(DISTINCT user_id) AS user_count,
    COUNT(DISTINCT session_id) AS session_count,
    ROUND(AVG(duration_min), 2) AS avg_duration_min
FROM raw_usage
GROUP BY access_platform
ORDER BY user_count DESC;"""

con.execute(platform).df()



,access_platform,user_count,session_count,avg_duration_min
0,Safari,868485,10283706,134.11
1,iOS,868438,10276616,134.27
2,Chrome,868134,10279880,134.32
3,Edge,868126,10278204,134.51
4,Android,868110,10293629,134.35
5,Firefox,867677,10273840,134.29
6,Opera,867652,10285543,134.34


In [10]:
# -- 3. Crash analysis by app
crash = """
SELECT 
    app_name,
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(DISTINCT session_id) FILTER (WHERE has_crash_ended_flag = 'Y' OR crash_time IS NOT NULL) AS crashed_sessions,
    ROUND(100.0 * COUNT(DISTINCT session_id) FILTER (WHERE has_crash_ended_flag = 'Y' OR crash_time IS NOT NULL) / COUNT(DISTINCT session_id), 2) AS crash_rate_pct
FROM raw_usage
GROUP BY app_name
ORDER BY crash_rate_pct DESC;""" 

con.execute(crash).df()

,app_name,total_sessions,crashed_sessions,crash_rate_pct
0,TikTok,2481483,1241381,50.03
1,Health,2481615,1241498,50.03
2,Photo,2474963,1238170,50.03
3,PlayStore,2479778,1240319,50.02
4,Weather,2490976,1246003,50.02
5,Flipkart,2484127,1242606,50.02
6,Snapchat,2493025,1247116,50.02
7,WhatsApp,2498377,1249566,50.02
8,VoiceRecorder,2483777,1242353,50.02
9,Wallet,2476610,1238733,50.02


In [14]:
import duckdb

# 1. Close the stale connection to release broken file handles
try:
    con.close()
except:
    pass

# 2. Start a fresh connection and establish safe runtime settings
con = duckdb.connect()
con.execute("SET preserve_insertion_order = false;")
con.execute("SET memory_limit = '8GB';")

# 3. Re-create the view pointing to your parquet data
con.execute(
    """
    CREATE OR REPLACE VIEW raw_usage AS 
    SELECT * FROM 'data/parquet/*.parquet';
"""
)

In [15]:
with open("sql/cleaning.sql", "r") as f:
    clean_query = f.read().strip().rstrip(";")

copy_sql = f"""
COPY (
    {clean_query}
) TO 'data/parquet/daily_app_metrics.parquet' (FORMAT PARQUET, CODEC 'ZSTD');
"""

print("Aggregating directly to Parquet...")
con.execute(copy_sql)
print("Complete!")

# Inspect generated row count
count = con.execute(
    "SELECT count(*) FROM 'data/parquet/daily_app_metrics.parquet'"
).fetchone()[0]
print(f"Final daily metrics rows: {count:,}")

Aggregating directly to Parquet...
Complete!
Final daily metrics rows: 6,344,206
